# AlphaLens Point-in-Time Feature Analysis

This notebook builds the model-input matrix from information available by each event's feature-as-of date. It examines feature coverage and univariate relationships without fitting a predictive model.

Feature families:

- trailing momentum, volatility, moving-average, volume, drawdown, beta, and SPY-relative signals;
- event type, fiscal quarter, calendar seasonality, and event spacing;
- token-weighted FinBERT sentiment, management-versus-analyst tone, and prior-event change;
- deterministic topic-level FinBERT aggregates for ten financial topics.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.features import (
    build_event_feature_dataset,
    model_feature_columns,
    summarize_feature_coverage,
    validate_feature_dataset,
)

pd.set_option('display.max_columns', 80)
sns.set_theme(style='whitegrid', context='notebook')

## Build the feature matrix

The model will receive only the explicit feature allowlist. Identifiers, future target dates, future prices, and the label itself are never selected as predictors.

In [ ]:
dataset = build_event_feature_dataset(horizon=30, include_topics=True)
features = list(model_feature_columns(include_topics=True))
checks = validate_feature_dataset(dataset, include_topics=True)
print(f'{len(dataset):,} events | {int(dataset.target_available.sum()):,} labeled | {len(features)} features')
checks

In [ ]:
identity = ['event_key', 'ticker', 'event_date', 'feature_as_of_date', 'event_source']
display(dataset[identity + features[:12] + ['excess_return_30d']].tail(8))

## Feature coverage

Some missingness is structural rather than erroneous. Filing rows have no management/analyst split, call rows have no 10-K/10-Q subtype, and a topic score is absent when that topic was not discussed. Later preprocessing must preserve these distinctions.

In [ ]:
coverage = summarize_feature_coverage(dataset, include_topics=True)
display(coverage.style.format({'coverage': '{:.1%}'}))

In [ ]:
plot_coverage = coverage.sort_values('coverage').tail(35)
fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(data=plot_coverage, x='coverage', y='feature', color='#00a67e', ax=ax)
ax.set(title='Feature coverage among labeled events', xlabel='Non-null coverage', ylabel='')
ax.xaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
ax.set_xlim(0, 1.02)
plt.tight_layout()
plt.show()

## Market features at the event anchor

Every market indicator is trailing and evaluated at `feature_as_of_date`, which must equal the post-event anchor session. Future-price mutation tests verify that later prices cannot alter these values.

In [ ]:
market_columns = [
    'momentum_21d', 'momentum_63d', 'volatility_21d',
    'sma_ratio_50d', 'volume_zscore_20d', 'drawdown_126d',
    'beta_63d', 'relative_momentum_63d',
]
display(dataset.loc[dataset.target_available, market_columns].describe().T)

## Univariate relationship with the label

These correlations are exploratory, not evidence of out-of-sample predictiveness. Model selection will use chronological validation with purged boundaries.

In [ ]:
labeled = dataset.loc[dataset.target_available].copy()
correlations = (
    labeled[features + ['excess_return_30d']]
    .corr(numeric_only=True)['excess_return_30d']
    .drop('excess_return_30d')
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
display(correlations.head(20).rename('pearson_correlation').to_frame())

In [ ]:
top_correlations = correlations.head(15).sort_values()
colors = ['#dc2626' if value < 0 else '#00a67e' for value in top_correlations]
ax = top_correlations.plot(kind='barh', figsize=(10, 6), color=colors)
ax.axvline(0, color='#111827', linewidth=1)
ax.set(title='Largest in-sample feature correlations', xlabel='Pearson correlation', ylabel='')
plt.tight_layout()
plt.show()

## FinBERT features

Event sentiment is token weighted. Earnings calls additionally expose management tone, analyst tone, and their gap. Topic sentiment uses the existing explainable keyword taxonomy to group FinBERT-scored text; the topic classifier itself is not FinBERT.

In [ ]:
sentiment_rows = labeled.dropna(subset=['sentiment_score']).copy()
sentiment_rows['sentiment_quintile'] = pd.qcut(
    sentiment_rows['sentiment_score'], 5, labels=False, duplicates='drop'
) + 1
sentiment_quintiles = (
    sentiment_rows.groupby(['event_source', 'sentiment_quintile'], as_index=False)
    .agg(
        events=('event_key', 'size'),
        mean_sentiment=('sentiment_score', 'mean'),
        mean_excess_return=('excess_return_30d', 'mean'),
    )
)
display(sentiment_quintiles.style.format({
    'mean_sentiment': '{:.3f}',
    'mean_excess_return': '{:.2%}',
}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(
    data=sentiment_quintiles, x='sentiment_quintile',
    y='mean_excess_return', hue='event_source', marker='o', ax=ax
)
ax.axhline(0, color='#111827', linewidth=1)
ax.set(title='Average excess return by FinBERT sentiment quintile', xlabel='Sentiment quintile', ylabel='Mean 30-day excess return')
ax.yaxis.set_major_formatter(lambda value, _: f'{value:.1%}')
plt.tight_layout()
plt.show()

## Decision for the next stage

The feature matrix is ready for baseline modeling. The next step is to define chronological train, validation, and test periods with a 30-session purge; fit naive and linear baselines; and record MAE, RMSE, directional accuracy, and rank correlation before introducing XGBoost.